# 23/ Handwritten Digit Recognition

**Dataset (Kaggle):** [https://www.kaggle.com/datasets/oddrationale/mnist-in-csv](https://www.kaggle.com/datasets/oddrationale/mnist-in-csv)

> MNIST in CSV form: 60,000 train / 10,000 test 28x28 grayscale digit images (0-9), one pixel per column.

**Goal:** Classify a handwritten digit image (0-9).

**How to get the data:**
1. Download from the link above (or run `kaggle datasets download -d <dataset-slug> -p data/ --unzip`, requires a `kaggle.json` API token in `~/.kaggle/`).
2. Place the extracted file(s) in a local `data/` folder next to this notebook.
3. Update `DATA_PATH` in the cell below if your filename differs.


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow.keras import layers, models
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix
import seaborn as sns

sns.set_style("whitegrid")
print("TF version:", tf.__version__)


In [ ]:
train_df = pd.read_csv("data/mnist_train.csv")
test_df = pd.read_csv("data/mnist_test.csv")

y_train = train_df['label'].values
X_train = train_df.drop(columns=['label']).values.reshape(-1, 28, 28, 1)
y_test = test_df['label'].values
X_test = test_df.drop(columns=['label']).values.reshape(-1, 28, 28, 1)
print(X_train.shape, X_test.shape)


## Preview Some Samples

In [ ]:
fig, axes = plt.subplots(2, 5, figsize=(12, 5))
for i, ax in enumerate(axes.flat):
    ax.imshow(X_train[i].squeeze(), cmap='gray' if X_train[i].shape[-1] == 1 else None)
    ax.set_title(str(y_train[i]))
    ax.axis('off')
plt.tight_layout()
plt.show()


## Preprocessing

In [ ]:
X_train = X_train.astype('float32') / 255.0
X_test = X_test.astype('float32') / 255.0

y_train_cat = tf.keras.utils.to_categorical(y_train, num_classes=10)
y_test_cat = tf.keras.utils.to_categorical(y_test, num_classes=10)

X_train, X_val, y_train_cat, y_val_cat = train_test_split(
    X_train, y_train_cat, test_size=0.1, random_state=42
)
print(X_train.shape, X_val.shape, X_test.shape)


## Build CNN Model

In [ ]:
model = models.Sequential([
    layers.Input(shape=(28, 28, 1)),
    layers.Conv2D(32, (3,3), activation='relu', padding='same'),
    layers.BatchNormalization(),
    layers.MaxPooling2D(2,2),
    layers.Conv2D(64, (3,3), activation='relu', padding='same'),
    layers.BatchNormalization(),
    layers.MaxPooling2D(2,2),
    layers.Conv2D(128, (3,3), activation='relu', padding='same'),
    layers.MaxPooling2D(2,2),
    layers.Flatten(),
    layers.Dense(128, activation='relu'),
    layers.Dropout(0.4),
    layers.Dense(10, activation='softmax')
])
model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])
model.summary()


## Train

In [ ]:
early_stop = tf.keras.callbacks.EarlyStopping(patience=4, restore_best_weights=True)

history = model.fit(
    X_train, y_train_cat,
    validation_data=(X_val, y_val_cat),
    epochs=20,
    batch_size=64,
    callbacks=[early_stop]
)


## Training Curves

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(12,4))
ax[0].plot(history.history['loss'], label='train'); ax[0].plot(history.history['val_loss'], label='val')
ax[0].set_title("Loss"); ax[0].legend()
ax[1].plot(history.history['accuracy'], label='train'); ax[1].plot(history.history['val_accuracy'], label='val')
ax[1].set_title("Accuracy"); ax[1].legend()
plt.show()


## Evaluation

In [ ]:
test_loss, test_acc = model.evaluate(X_test, y_test_cat)
print(f"Test accuracy: {test_acc:.4f}")

y_pred = np.argmax(model.predict(X_test), axis=1)
print(classification_report(y_test, y_pred))

cm = confusion_matrix(y_test, y_pred)
plt.figure(figsize=(8,7))
sns.heatmap(cm, annot=False, cmap='Blues')
plt.xlabel("Predicted"); plt.ylabel("Actual")
plt.title("Confusion Matrix")
plt.show()


## Notes
The classic 'hello world' of computer vision — a great sanity check before moving to harder image tasks.

## Next Steps
- Add data augmentation (`ImageDataGenerator` / `tf.keras.layers.RandomFlip`, `RandomRotation`).
- Try transfer learning with `MobileNetV2` or `ResNet50` pretrained on ImageNet.
- Export with `model.save('model.keras')` and serve via TensorFlow Serving or ONNX.